### Configuration Settings 

In [1]:
# Create config.py file
config_content = '''import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Data Source
INFORMATICS_URL = "https://twu.edu/informatics/graduate-program/"

# Database Settings
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "twu-ai-chatbot"
EMBEDDING_MODEL = "multilingual-e5-large"

# Processing Settings
CHUNK_SIZE = 120 
CHUNK_OVERLAP = 15
TOP_K_RESULTS = 3

# API Settings

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

VERTEX_API_KEY = os.getenv("VERTEX_API_KEY")
'''

with open('config.py', 'w', encoding='utf-8') as f:
    f.write(config_content)

print("Config.py File Created ")

Config.py File Created 


In [2]:
import os 
print("config.py exists:", os.path.exists('config.py'))

# Preview the file
with open('config.py', 'r') as f:
    print(f.read())

config.py exists: True
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Data Source
INFORMATICS_URL = "https://twu.edu/informatics/graduate-program/"

# Database Settings
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = "twu-ai-chatbot"
EMBEDDING_MODEL = "multilingual-e5-large"

# Processing Settings
CHUNK_SIZE = 120 
CHUNK_OVERLAP = 15
TOP_K_RESULTS = 3

# API Settings

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

VERTEX_API_KEY = os.getenv("VERTEX_API_KEY")



### Cleaning and Scrapping Webpage 

In [3]:
# Create scraper.py file 
scraper_content = """import requests
from bs4 import BeautifulSoup
import config
import re

def get_clean_text():
    \"\"\"Scrape and clean text from TWU webpage - captures ALL content\"\"\"
    
    # Get the webpage
    response = requests.get(config.INFORMATICS_URL)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Remove only pure code elements (not content)
    if soup.head:
        soup.head.decompose()
    
    for tag in soup.find_all(['script', 'style', 'noscript']):
        tag.decompose()
    
    # Collect text from ALL important sections
    text_parts = []
    
    # Main content 
    main_content = soup.find('main')
    if main_content:
        text_parts.append(main_content.get_text())
    
    # Sidebar 
    sidebar = soup.find('div', class_='sidebar')
    if sidebar:
        text_parts.append(sidebar.get_text())
        
    # Accordion sections
    accordions = soup.find_all('div', {'data-aria-accordion': True})
    for accordion in accordions:
        text_parts.append(accordion.get_text())
    
    # Jump-scroll divs (contain deadlines and contact)
    jump_scrolls = soup.find_all('div', class_='jump-scroll')
    for js in jump_scrolls:
        text_parts.append(js.get_text())
    
    # Feature boxes 
    feature_boxes = soup.find_all('div', class_='feature')
    for feature in feature_boxes:
        text_parts.append(feature.get_text())
    
    # Combine all parts
    raw_text = '\\n'.join(text_parts)
    
    # Clean up the text 
    lines = []
    for line in raw_text.split('\\n'):
        line = line.strip()
        # Keep lines that have meaningful content
        if line and len(line) > 2:
            lines.append(line)
    
    return '\\n'.join(lines)

def chunk_text(text):
    \"\"\"Split text into chunks\"\"\"
    
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    import tiktoken
    
    tokenizer = tiktoken.get_encoding("cl100k_base")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        length_function=lambda x: len(tokenizer.encode(x)),
        separators=["\\n\\n", "\\n", ". ", " ", ""]
    )
    
    return splitter.split_text(text)
"""

with open('scraper.py', 'w', encoding='utf-8') as f:
    f.write(scraper_content)

### Creating Chunks for RAG 

In [4]:
import sys
sys.path.append('.')

from scraper import get_clean_text, chunk_text

print(f"get_clean_text function: {get_clean_text}")
print(f"chunk_text function: {chunk_text}")

get_clean_text function: <function get_clean_text at 0x13b29ee80>
chunk_text function: <function chunk_text at 0x13b29ef20>


In [5]:
# Test scraping
print("Testing get_clean_text:")
clean_text = get_clean_text()
print(f" {len(clean_text)} characters")
print(f"Preview: {clean_text[:200]}...")

# Test chunking
print("\nTesting chunk_text:")
chunks = chunk_text(clean_text)
print(f" {len(chunks)} chunks")
if chunks:
    print(f"First chunk: {chunks[0][:400]}...")

Testing get_clean_text:
 13752 characters
Preview: Data Science & Informatics Graduate Program
Master of Science in Data Science & Informatics
Complete Coursework: 18 months* | Credit Hours: 30** | Format: online or hybrid
Turn big data into big ideas...

Testing chunk_text:
 28 chunks
First chunk: Data Science & Informatics Graduate Program
Master of Science in Data Science & Informatics
Complete Coursework: 18 months* | Credit Hours: 30** | Format: online or hybrid
Turn big data into big ideas with our inter-professional master’s degree Data Science & Informatics. Classes offered in online and in-person formats. Tailor your education to your interests by choosing from an emphasis in:
Clini...


### Pinecone Vector Database 

In [6]:
# Create database.py file

database_content = '''import pinecone
from pinecone import ServerlessSpec
import config
import time

class VectorStore:
    def __init__(self):
        self.pc = pinecone.Pinecone(api_key=config.PINECONE_API_KEY)
        self.index_name = config.INDEX_NAME
        self.index = None
    
    def create_index(self):
        """Create a new index (deletes existing)"""
        
        # Delete if exists 
        if self.index_name in self.pc.list_indexes().names():
            self.pc.delete_index(self.index_name)
            time.sleep(5)
        
        # Create new index
        self.pc.create_index(
            name=self.index_name,
            dimension=1024,
            metric="cosine",
            spec=ServerlessSpec(cloud='aws', region='us-east-1')
        )
        
        while not self.pc.describe_index(self.index_name).status['ready']:
            time.sleep(1)
        
        self.index = self.pc.Index(self.index_name)
        print(f" Index '{self.index_name}' ready")
    
    def connect_index(self):
        """Connect to existing index """
        
        if self.index_name not in self.pc.list_indexes().names():
            print(f" Index '{self.index_name}' does not exist. Call create_index() first.")
            return False
        
        self.index = self.pc.Index(self.index_name)
        print(f" Connected to existing index '{self.index_name}'")
        return True
    
    def upload_chunks(self, chunks):
        """Upload chunks to Pinecone"""
        
        if not self.index:
            print(" No index connection. Call connect_index() or create_index() first.")
            return
        
        vectors = []
        for i, chunk in enumerate(chunks):
            embedding = self.pc.inference.embed(
                model=config.EMBEDDING_MODEL,
                inputs=[chunk],
                parameters={"input_type": "passage"}
            )
            
            vectors.append({
                'id': f'chunk_{i:03d}',
                'values': embedding[0].values,
                'metadata': {'text': chunk}
            })
        
        # Upload in batches
        batch_size = 10
        for i in range(0, len(vectors), batch_size):
            batch = vectors[i:i+batch_size]
            self.index.upsert(vectors=batch)
        
        print(f" Chunks {len(chunks)} Uploaded")
    
    def query(self, query_text, top_k=config.TOP_K_RESULTS):
        """Search for relevant chunks"""
        
        if not self.index:
            print(" No index connection. Call connect_index() first.")
            return []
        
        # Generate query embedding
        embedding = self.pc.inference.embed(
            model=config.EMBEDDING_MODEL,
            inputs=[query_text],
            parameters={"input_type": "query"}
        )
        
        # Search
        results = self.index.query(
            vector=embedding[0].values,
            top_k=top_k,
            include_metadata=True
        )
        
        chunks = []
        for match in results['matches']:
            chunks.append({
                'text': match['metadata']['text'],
                'score': match['score']
            })
        
        return chunks
'''

with open('database.py', 'w', encoding='utf-8') as f:
    f.write(database_content)

print(" Database.py Created ")

 Database.py Created 


### RAG Utilization and Testing 

In [16]:
import sys
import os
sys.path.append('.')

from scraper import get_clean_text, chunk_text
from database import VectorStore
from google import genai
import config

# Initialize Gemini
os.environ["GEMINI_API_KEY"] = config.GEMINI_API_KEY
gemini_client = genai.Client()

# Generative Answer 
def generate_answer(query, chunks):
    """Generate answer using Gemini"""
    if not chunks:
        return "No relevant information found in the database."
    
    context = "\n\n---\n\n".join([chunk['text'] for chunk in chunks[:3]])   
    prompt = f"""You are a TWU Informatics program assistant. Answer using ONLY the context below. If the answer is not in the context, say "I don't have that information."

CONTEXT: {context}
QUESTION: {query}
ANSWER:"""
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text

# Get chunks
clean_text = get_clean_text()
chunks = chunk_text(clean_text)
print(f"Created {len(chunks)} Chunks")

# Setup VectorStore
db = VectorStore()
db.create_index()
db.upload_chunks(chunks)

# Test questions 
test_questions = [
    "What are the application deadlines?",
    "What are the GPA requirements?",
    "What degree plans are offered in the program?",
    "What is the application fee?"
]

for query in test_questions:
    print(f"\n Question: {query}")
    
    # Retrieve from Pinecone
    retrieved = db.query(query)
    print(f"   Retrieved {len(retrieved)} chunks")
    print(f"   Best score: {retrieved[0]['score']:.3f}")
    
    # Generate answer with Gemini
    answer = generate_answer(query, retrieved)
    print(f"\n Answer: {answer}")
    

Created 28 Chunks
 Index 'twu-ai-chatbot' ready
 Chunks 28 Uploaded

 Question: What are the application deadlines?
   Retrieved 3 chunks
   Best score: 0.827

 Answer: Fall Admission - August 1
Spring Admission - December 10

 Question: What are the GPA requirements?
   Retrieved 3 chunks
   Best score: 0.850

 Answer: Minimum 3.0 GPA in the last 60 credit hours

 Question: What degree plans are offered in the program?
   Retrieved 3 chunks
   Best score: 0.782

 Answer: Data Science & Informatics Clinical applications MS
Data Science & Informatics Community MS
Data Science & Informatics Cybersecurity MS
Data Science & Informatics/data analytics MS

 Question: What is the application fee?
   Retrieved 3 chunks
   Best score: 0.832

 Answer: The application fee is $50 ($75 for International applicants).


### Comparison of Google Gemini vs Vertex API 

In [17]:
import os
import time
import statistics
from google import genai
import config
from database import VectorStore


# Initialize Pinecone 
db = VectorStore()
db.connect_index()

# Test questions  (same as previous) 
test_questions = [
    "What are the application deadlines?",
    "What are the GPA requirements?",
    "What degree plans are offered in the program?",
    "What is the application fee?"
]

# Initialize both Gemini and Vertex AI clients
gemini_client = genai.Client(api_key=config.GEMINI_API_KEY)
vertex_client = genai.Client(vertexai=True, api_key=config.VERTEX_API_KEY)

def generate_answer_with_context(client, query, chunks):
    """Generate answer using either Gemini or Vertex AI with context"""
    if not chunks:
        return "No relevant information found."
    
    context = "\n\n---\n\n".join([chunk['text'] for chunk in chunks[:3]])
    prompt = f"""Answer based on the context below. If unsure, say "I don't know."

CONTEXT: {context}
QUESTION: {query}
ANSWER:"""
    
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text

results = {"RAG with Gemini": [], "RAG with Vertex AI": []}

for api_name, client in [("RAG with Gemini", gemini_client), 
                          ("RAG with Vertex AI", vertex_client)]:
    
    print(f"\n{api_name}")
    print("-" * 20)
    
    response_times = []
    
    for question in test_questions:
        #  Retrieve Pinecone Information 
        retrieved = db.query(question)
        
        # Generation 
        start = time.time()
        answer = generate_answer_with_context(client, question, retrieved)
        elapsed = time.time() - start
        response_times.append(elapsed)
        
        print(f"\nQ: {question}")
        print(f"   Retrieved: {len(retrieved)} chunks")
        if retrieved:
            print(f"   Best score: {retrieved[0]['score']:.3f}")
        print(f"   Time: {elapsed:.2f}s")
        print(f"   Answer: {answer[:200]}...")
        
        results[api_name].append({"time": elapsed, "answer": answer})
    
    avg_time = statistics.mean(response_times)
    print(f"\n{api_name} - Avg Time: {avg_time:.2f}s")



gem_avg = statistics.mean([r["time"] for r in results["RAG with Gemini"]])
vert_avg = statistics.mean([r["time"] for r in results["RAG with Vertex AI"]])

print(f"RAG with Gemini:    {gem_avg:.2f} seconds")
print(f"RAG with Vertex AI: {vert_avg:.2f} seconds")
print(f"Difference: {abs(gem_avg - vert_avg):.2f} seconds")

if gem_avg < vert_avg:
    print("\nGemini was faster")
else:
    print("\nVertex AI was faster")

 Connected to existing index 'twu-ai-chatbot'

RAG with Gemini
--------------------

Q: What are the application deadlines?
   Retrieved: 3 chunks
   Best score: 0.827
   Time: 0.50s
   Answer: Fall Admission - August 1
Spring Admission - December 10...

Q: What are the GPA requirements?
   Retrieved: 3 chunks
   Best score: 0.850
   Time: 0.53s
   Answer: Minimum 3.0 GPA in the last 60 credit hours...

Q: What degree plans are offered in the program?
   Retrieved: 3 chunks
   Best score: 0.782
   Time: 0.46s
   Answer: The following degree plans are offered in the program:
* Data Science & Informatics Clinical applications MS
* Data Science & Informatics Community MS
* Data Science & Informatics Cybersecurity MS
* D...

Q: What is the application fee?
   Retrieved: 3 chunks
   Best score: 0.832
   Time: 1.07s
   Answer: $75 fee for International applicants...

RAG with Gemini - Avg Time: 0.64s

RAG with Vertex AI
--------------------

Q: What are the application deadlines?
   Retrieve

### Building Execution Script 

In [18]:
# Create main.py file
main_content = '''import os
import sys
from google import genai
import config
from scraper import get_clean_text, chunk_text
from database import VectorStore

# Initialize Gemini
os.environ["GEMINI_API_KEY"] = config.GEMINI_API_KEY
gemini_client = genai.Client()

def generate_answer(query, chunks):
    """Generate answer using Gemini"""
    
    context = "\\n\\n---\\n\\n".join([chunk['text'] for chunk in chunks])
    
    prompt = f"""Answer based on the context below. If unsure, say "I don't know."

CONTEXT: {context}
QUESTION: {query}
ANSWER:"""
    
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt)
    
    return response.text

def main():
    print(" TWU Informatics Webpage RAG System")
    print("=" * 30)
    
    # Step 1: Scrape and clean
    clean_text = get_clean_text()
    print(f"   Cleaned {len(clean_text)} characters")
    
    # Step 2: Chunk
    chunks = chunk_text(clean_text)
    print(f"   Created {len(chunks)} chunks")
    
    # Step 3: Upload to Pinecone
    db = VectorStore()
    db.create_index() #create the index 1st 
    db.upload_chunks(chunks)
    
    # Step 4: Test queries
    test_questions = [
        "What are the application deadlines?",
        "What are the GPA requirements?"
    ]
    
    for q in test_questions:
        print(f"\\n Question: {q}")
        retrieved = db.query(q)
        print(f"   Best score: {retrieved[0]['score']:.3f}")
        answer = generate_answer(q, retrieved)
        print(f" Answer: {answer}")
    
    print("RAG System Complete!")
    print("=" * 50)

if __name__ == "__main__":
    main()
'''

with open('main.py', 'w', encoding='utf-8') as f:
    f.write(main_content)

print(" Main.py Created")

 Main.py Created


### Requirements File 

In [19]:
# Create requirements.txt file
requirements_content = '''pinecone
google-generativeai
beautifulsoup4
requests
langchain-text-splitters
tiktoken
python-dotenv
'''

with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_content)

print(" Requirements.txt Created")

 Requirements.txt Created


### Docker File 

In [20]:
# Create Dockerfile
dockerfile_content = '''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python", "main.py"]
'''

with open('Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile_content)

print(" Dockerfile Created")

 Dockerfile Created


In [23]:
# Create .dockerignore file
dockerignore_content = '''.env
__pycache__
*.pyc
.DS_Store
.git
*.ipynb
'''

with open('.dockerignore', 'w', encoding='utf-8') as f:
    f.write(dockerignore_content)

print(" .dockerignore Created")

 .dockerignore Created


In [24]:
# Create .gitignore file 

gitignore_content = '''.env

# Python cache files
__pycache__/
*.pyc
*.pyo

# Jupyter notebooks 
.ipynb_checkpoints/

# Mac system files
.DS_Store

# IDE files
.vscode/
.idea/

# Docker
*.log
'''

with open('.gitignore', 'w', encoding='utf-8') as f:
    f.write(gitignore_content)

print(".gitignore created ")
print(gitignore_content)

.gitignore created 
.env

# Python cache files
__pycache__/
*.pyc
*.pyo

# Jupyter notebooks 
.ipynb_checkpoints/

# Mac system files
.DS_Store

# IDE files
.vscode/
.idea/

# Docker
*.log

